# Nettoyage du domaine public records

Le domaine contient 9 fichiers. Le nettoyage a été fait en deux parties :
- partie Yannis : `site_access_case_events`, `regulatory_case_events`, `registered_entities`, `public_appointments`, `public_and_investigative_documents` ;
- partie Dali : `institutional_memos`, `licensed_facilities`, `licensed_professionals`, `officer_filings`.

On garde leur code. Les corrections sont signalées par une cellule « Correction ». À la fin, les 9 fichiers propres sont enregistrés dans `data_clean/public_records/`, sous leur nom d'origine.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

# pour l'inspection c'est + clair affiché comme ça
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

SOURCE = Path("data/public_records")
SORTIE = Path("data_clean/public_records")

## Partie Yannis

### Nettoyage global

Pour les 5 fichiers : suppression des doublons, espaces en trop, valeurs manquantes écrites de plusieurs façons, catégories en minuscules et dates.

**Correction** : la variable qui contenait `registered_entities` s'appelait `re`, comme la bibliothèque Python `re` (expressions régulières) utilisée plus bas dans la partie Dali. Les deux se seraient écrasées : on la renomme `ent`.

In [2]:
sace = pd.read_csv(SOURCE / "site_access_case_events.csv")
rce = pd.read_csv(SOURCE / "regulatory_case_events.csv")
ent = pd.read_csv(SOURCE / "registered_entities.csv")
pa = pd.read_csv(SOURCE / "public_appointments.csv")
paid = pd.read_csv(SOURCE / "public_and_investigative_documents.csv")

dfs = {
    "site_access": sace,
    "regulatory_cases": rce,
    "registered_entities": ent,
    "public_appointments": pa,
    "investigative_docs": paid
}

# standardisation en minuscules
cat_cols = {
    "activity", "case_category", "confidentiality_class",
    "previous_status", "new_status", "registered_entity_type",
    "current_status", "appointment_basis", "departure_reason",
    "official_region", "public_or_restricted", "jurisdiction"
}

for name, df in dfs.items():
    # Supp des doublons stricts
    nb_dups = df.duplicated().sum()
    if nb_dups > 0:
        print(f"[{name}] {nb_dups} doublons")
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)

    # espaces superflus + normalisation des valeurs manquantes
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        if col == "text":
            # pour garder la structure en paragraphes (pour le textmining)
            df[col] = df[col].astype(str).str.replace("\r\n", "\n").str.replace(r"[ \t]+", " ", regex=True).str.strip()
        else:
            df[col] = df[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

        # minuscules
        if col in cat_cols:
            df[col] = df[col].str.lower()

    # Harmonisation + typage des dates
    date_cols = [c for c in df.columns if any(k in c.lower() for k in ["date", "time", "start", "end"])]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce", format="mixed")

print("nettoyage global okay")

[site_access] 1 doublons
[regulatory_cases] 1 doublons
nettoyage global okay


On vérifie qu'aucune date n'a été perdue pendant la conversion : une date illisible devient une case vide.

In [3]:
noms_origine = {"site_access": "site_access_case_events", "regulatory_cases": "regulatory_case_events",
                "registered_entities": "registered_entities", "public_appointments": "public_appointments",
                "investigative_docs": "public_and_investigative_documents"}

for name, df in dfs.items():
    brut = pd.read_csv(SOURCE / f"{noms_origine[name]}.csv", dtype=str).drop_duplicates()
    for col in df.select_dtypes(include="datetime").columns:
        perdues = brut[col].notna().sum() - df[col].notna().sum()
        print(f"{name:22s} {col:20s} dates perdues : {perdues}")

site_access            event_time           dates perdues : 0
regulatory_cases       event_time           dates perdues : 0
registered_entities    registration_date    dates perdues : 0
registered_entities    last_filing_date     dates perdues : 0
public_appointments    appointment_start    dates perdues : 0
public_appointments    appointment_end      dates perdues : 0
investigative_docs     document_date        dates perdues : 0


Aucune date n'est perdue.

### Traitements propres à certains fichiers

- Dans `public_appointments`, une même personne apparaît sur plusieurs lignes, et son année de naissance manque parfois : on la complète à partir de ses autres lignes.
- On calcule l'âge à la prise de fonction, et on repère les mandats encore en cours (pas de date de fin).
- On trie les dossiers par numéro et par date, pour garder l'ordre des étapes.

In [4]:
# public_appointments
# Certains individus apparaissent plusieurs fois avec parfois leur année de naissance manquante..
pa["recorded_birth_year"] = pa.groupby("office_holder_name")["recorded_birth_year"].transform(
    lambda s: s.bfill().ffill()
)

# Calcul de l'âge lors de la prise de fonction et  mandats en cours
pa["age_at_appointment"] = pa["appointment_start"].dt.year - pa["recorded_birth_year"]
pa["is_active_mandate"] = pa["appointment_end"].isna()

pa["recorded_birth_year"] = pa["recorded_birth_year"].astype("Int64")
pa["age_at_appointment"] = pa["age_at_appointment"].astype("Int64")

# Tri chronologique (ça peut être utile pour les analyses suivantes)
rce.sort_values(by=["case_number", "event_time"], inplace=True)
rce.reset_index(drop=True, inplace=True)

sace.sort_values(by=["access_case_number", "event_time"], inplace=True)
sace.reset_index(drop=True, inplace=True)

# registered_entities
# parent_registration_number est vide à 100%, mais on la garde :
# l'absence de société-mère déclarée peut être utile ensuite

In [5]:
#Juste pour checker les dimensions
for name, df in dfs.items():
    print(f"{name:<22} : {df.shape[0]} lignes, {df.shape[1]} colonnes")

site_access            : 412 lignes, 15 colonnes
regulatory_cases       : 232 lignes, 16 colonnes
registered_entities    : 24 lignes, 10 colonnes
public_appointments    : 62 lignes, 14 colonnes
investigative_docs     : 40 lignes, 11 colonnes


### Export

**Correction** : l'export était en commentaire, donc aucun fichier propre n'était enregistré. On l'active, et on enregistre dans `data_clean/public_records/` sous les noms d'origine, comme pour les autres domaines.

In [6]:
SORTIE.mkdir(parents=True, exist_ok=True)

fichiers_yannis = {
    "site_access_case_events": sace,
    "regulatory_case_events": rce,
    "registered_entities": ent,
    "public_appointments": pa,
    "public_and_investigative_documents": paid,
}
for nom, df in fichiers_yannis.items():
    df.to_csv(SORTIE / f"{nom}.csv", index=False)
    print(f"Enregistré : {nom}.csv ({df.shape[0]} lignes, {df.shape[1]} colonnes)")

Enregistré : site_access_case_events.csv (412 lignes, 15 colonnes)
Enregistré : regulatory_case_events.csv (232 lignes, 16 colonnes)
Enregistré : registered_entities.csv (24 lignes, 10 colonnes)
Enregistré : public_appointments.csv (62 lignes, 14 colonnes)
Enregistré : public_and_investigative_documents.csv (40 lignes, 11 colonnes)


## Partie Dali

On charge les 4 fichiers et on regarde les doublons et les valeurs manquantes avant de toucher à quoi que ce soit.

**Correction** : les chemins commençaient par `../data/`, parce que le notebook était rangé dans un sous-dossier. Le notebook est maintenant à la racine du projet, on utilise donc `data/`.

In [7]:
im = pd.read_csv(SOURCE / "institutional_memos.csv")
lf = pd.read_csv(SOURCE / "licensed_facilities.csv")
lp = pd.read_csv(SOURCE / "licensed_professionals.csv")
of = pd.read_csv(SOURCE / "officer_filings.csv")

dfs = {
    "institutional_memos": im,
    "licensed_facilities": lf,
    "licensed_professionals": lp,
    "officer_filings": of
}

for name, df in dfs.items():
    print(f"--- {name} : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    print("doublons :", df.duplicated().sum())
    na = df.isna().sum()
    print("NaN :", na[na > 0].to_dict())
    print()

--- institutional_memos : 30 lignes, 12 colonnes
doublons : 0
NaN : {'related_purchase_order': 25, 'related_shipment_number': 22, 'related_case_number': 20}

--- licensed_facilities : 3 lignes, 12 colonnes
doublons : 0
NaN : {'license_end': 3}

--- licensed_professionals : 53 lignes, 10 colonnes
doublons : 0
NaN : {'license_expiry_date': 53, 'primary_facility_license': 26}

--- officer_filings : 136 lignes, 12 colonnes
doublons : 1
NaN : {'cessation_date': 77}



### Nettoyage global

Doublons, espaces en trop et dates.

Pour les dates, on lit d'abord le format standard `aaaa-mm-jj`, puis seulement les autres formats en considérant que le jour vient en premier. L'ordre compte : si on mettait `dayfirst=True` sur toutes les dates, pandas inverserait le jour et le mois des dates déjà bien écrites (le 5 février deviendrait le 2 mai).

In [8]:
for name, df in dfs.items():

    nb_dups = df.duplicated().sum()
    if nb_dups > 0:
        print(f"[{name}] {nb_dups} doublon(s) supprimé(s)")
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)

    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        avant = df[col].copy()
        if col == "text":

            df[col] = df[col].str.replace("\r\n", "\n").str.replace(r"[ \t]+", " ", regex=True).str.strip()
        else:
            df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()
            modifs = avant[(avant != df[col]) & avant.notna()]
            if len(modifs) > 0:
                print(f"[{name}] {col} : {modifs.tolist()}")

    date_cols = [c for c in df.columns if "date" in c or c in ["license_start", "license_end"]]
    for col in date_cols:
        d = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")
        d = d.fillna(pd.to_datetime(df[col], format="mixed", dayfirst=True, errors="coerce"))
        df[col] = d

print("nettoyage global okay")

[institutional_memos] subject_line : ['  Internal  Operations  Memorandum  57 ']
[licensed_professionals] specialty : ['  environmental  assessment ']
[officer_filings] 1 doublon(s) supprimé(s)
[officer_filings] full_name : ['  Pavel  Waren ']
nettoyage global okay


On vérifie deux dates qui étaient écrites dans un autre format.

In [9]:
print(lp.loc[lp["professional_name"] == "Doran Maren", "license_issue_date"])
print(of.loc[of["appointment_date"].dt.year == 1986, ["full_name", "appointment_date"]])

26   1994-07-01
Name: license_issue_date, dtype: datetime64[us]
       full_name appointment_date
57    Faris Uren       1986-01-01
59    Kira Norel       1986-01-01
133  Miran Istra       1986-01-01


### Traitements propres à chaque fichier

`institutional_memos` : les mémos sont écrits en sections (SUBJECT, CONTEXT, FACTS…). On ajoute le nombre de mots, et une colonne par section pour pouvoir les comparer plus facilement.

In [10]:
im.sort_values("document_date", inplace=True)
im.reset_index(drop=True, inplace=True)

im["nb_mots"] = im["text"].str.split().str.len()
im["visibility_level"] = pd.Categorical(im["visibility_level"], categories=["internal", "confidential", "restricted"], ordered=True)

re_titres = re.compile(r"^([A-Z][A-Z ]{2,})$", flags=re.M)

def decoupe_sections(text):
    parts = re_titres.split(text)
    return {parts[i].strip().lower().replace(" ", "_"): parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)}

sections = pd.DataFrame(im["text"].apply(decoupe_sections).tolist()).add_prefix("sec_")
im = pd.concat([im, sections], axis=1)

print(sections.notna().sum().sort_values(ascending=False))

sec_action                   11
sec_subject                   6
sec_context                   6
sec_facts                     5
sec_agenda                    5
sec_discussion                5
sec_decisions                 5
sec_scope                     5
sec_authorization             5
sec_supplier_and_quantity     5
sec_justification             5
sec_request                   5
sec_exception                 5
sec_cargo_and_timing          5
sec_escalation                5
sec_route                     5
sec_responsibilities          5
sec_period                    4
sec_reviewed_causes           4
sec_adjustment                4
sec_meeting_record            4
sec_discrepancy               4
sec_controls                  4
sec_legacy_system             1
sec_control_requirements      1
sec_findings                  1
sec_routing_note              1
sec_limits                    1
dtype: int64


`licensed_facilities` : on ajoute le nom de l'organisme qui gère chaque établissement, à partir de `registered_entities`.

**Correction** : la liste des services était transformée en liste Python. Dans un fichier CSV, une liste est enregistrée comme du texte (`"['a', 'b']"`) et n'est plus une vraie liste quand on relit le fichier. On garde donc le texte d'origine, avec les services séparés par `; `.

In [11]:
lf = lf.merge(ent[["registration_number", "legal_name"]], left_on="operating_organization_registration", right_on="registration_number", how="left")
lf.drop(columns="registration_number", inplace=True)
lf.rename(columns={"legal_name": "operating_organization_name"}, inplace=True)
lf

,facility_license_number,facility_name,facility_category,operating_organization_registration,street_address,region,latitude,longitude,license_start,license_end,facility_status,declared_services,operating_organization_name
0,FAC-VLN-101,Asterion Central Clinical Campus,research_hospital,REG-VLN-01002,"Asterion Central Clinical Campus, Valenne Cent...",Valenne Republic,47.20,7.51,2001-02-01,NaT,active,preventive medicine; cardiac assessment; clini...,Asterion Clinical Institute
1,FAC-AVL-201,Asterion Avelora Executive Clinic,private_clinic,REG-VLN-01002,"Asterion Avelora Executive Clinic, Eastern Ave...",Avelora Federation,7.35,101.62,2008-05-15,NaT,active,executive health; specialist follow-up; outpat...,Asterion Clinical Institute
2,FAC-AVL-301,Threshold Earth Avelora Laboratory,research_laboratory,REG-AVL-01007,"Threshold Earth Avelora Laboratory, Eastern Av...",Avelora Federation,7.50,101.75,2012-01-18,NaT,active,environmental health research; non-invasive hu...,Threshold Earth Observatory


`licensed_professionals` : on retire le titre « Dr. » des noms, pour pouvoir les comparer aux autres fichiers, et on calcule l'ancienneté de chaque licence au 1er janvier 2026.

On vérifie aussi que chaque établissement cité existe bien dans `licensed_facilities`.

In [12]:
lp["name_clean"] = lp["professional_name"].str.replace("Dr. ", "", regex=False)

ref_date = pd.Timestamp("2026-01-01")
lp["license_age_years"] = ((ref_date - lp["license_issue_date"]).dt.days / 365.25).round(1)

print(set(lp["primary_facility_license"].dropna()) - set(lf["facility_license_number"]))

print(lp.sort_values("license_issue_date")[["license_number", "name_clean", "profession", "specialty", "license_issue_date", "license_age_years", "primary_facility_license"]].head(6))

set()


      license_number    name_clean                          profession                 specialty license_issue_date  license_age_years primary_facility_license
6   LIC-KNT-RES-1007     J. Varlan               research professional      ecological analytics         1886-07-01              139.5              FAC-AVL-301
0   LIC-VLN-MED-1001   Emil Warden                           physician                cardiology         1895-07-01              130.5              FAC-VLN-101
7   LIC-AVL-RES-1008    Jonah Bell               research professional    public policy research         1924-07-01              101.5              FAC-AVL-301
12  LIC-AVL-ADM-1013    Elira Vaun  public administration professional  programme administration         1965-07-01               60.5                      NaN
8   LIC-VLN-ENV-1009  Julien Marot          environmental professional       wildlife monitoring         1970-07-01               55.5              FAC-AVL-301
49  LIC-VLN-ADM-1050   Jalen Havel  publ

Tous les établissements cités existent.

Une licence a une ancienneté négative : celle de Farah Havel, délivrée en juillet 2026, après la date de référence. Ce n'est pas une erreur de calcul, la date est simplement postérieure au 1er janvier 2026.

In [13]:
lp[lp["license_age_years"] < 0][["name_clean", "license_issue_date", "license_age_years"]]

,name_clean,license_issue_date,license_age_years
5,Farah Havel,2026-07-01,-0.5


`officer_filings` : l'année de naissance est parfois une tranche (« 1960–1969 »). On garde le minimum, le maximum et le milieu de la tranche. On calcule ensuite l'âge à la nomination, si le mandat est en cours, et sa durée.

On repère aussi les mandats qui se terminent avant d'avoir commencé.

In [14]:
annees = of["birth_year_or_age_band"].str.split("–", expand=True)
of["birth_year_min"] = annees[0].astype(int)
of["birth_year_max"] = annees[1].fillna(annees[0]).astype(int)
of["birth_year_est"] = (of["birth_year_min"] + of["birth_year_max"]) / 2
of["age_at_appointment"] = of["appointment_date"].dt.year - of["birth_year_est"]

of["is_active_mandate"] = of["cessation_date"].isna()
print(pd.crosstab(of["filing_status"], of["is_active_mandate"]))

of["mandate_duration_years"] = ((of["cessation_date"].fillna(ref_date) - of["appointment_date"]).dt.days / 365.25).round(1)

of["cessation_before_appointment"] = of["cessation_date"] < of["appointment_date"]
print("\nfin avant début :", of["cessation_before_appointment"].sum())
print(of[of["cessation_before_appointment"]][["filing_id", "full_name", "registration_number", "appointment_date", "cessation_date", "filing_date"]])

is_active_mandate  False  True 
filing_status                  
active                 0     77
historical            58      0

fin avant début : 9


      filing_id     full_name registration_number appointment_date cessation_date filing_date
35   FLG-000036    Calen Uren       REG-NDH-01011       2007-01-01     2006-12-31  2007-03-08
49   FLG-000050   Liora Corda       REG-VLN-01002       2014-01-01     2012-12-31  2014-02-23
76   FLG-000077  Jalen Quorin       REG-VLN-01002       2013-01-01     2011-12-31  2013-01-26
83   FLG-000084    Orin Norel       REG-VLN-01005       1998-01-01     1997-12-31  1998-01-26
99   FLG-000100   Pavel Waren       REG-VLN-01021       2011-01-01     2010-12-31  2011-02-03
123  FLG-000124   Elian Cevan       REG-VLN-01010       2025-01-01     2024-12-31  2025-03-04
126  FLG-000127   Daria Garen       REG-VLN-01021       2000-01-01     1998-12-31  2000-03-15
128  FLG-000129    Uma Wessen       REG-VLN-01023       2015-01-01     2014-12-31  2015-02-06
130  FLG-000131   Petra Koral       REG-VLN-01002       2005-01-01     2004-12-31  2005-03-01


- Le statut du dossier (`filing_status`) correspond bien aux mandats en cours ou terminés.
- 9 mandats se terminent avant d'avoir commencé (de quelques jours à 2 ans) : on les garde, mais on les signale dans la colonne `cessation_before_appointment`.

In [15]:
of.sort_values(["registration_number", "appointment_date"], inplace=True)
of.reset_index(drop=True, inplace=True)

dfs = {"institutional_memos": im, "licensed_facilities": lf, "licensed_professionals": lp, "officer_filings": of}
for name, df in dfs.items():
    print(f"{name:<24} : {df.shape[0]} lignes, {df.shape[1]} colonnes")

institutional_memos      : 30 lignes, 41 colonnes
licensed_facilities      : 3 lignes, 13 colonnes
licensed_professionals   : 53 lignes, 12 colonnes
officer_filings          : 135 lignes, 19 colonnes


### Export

Dans `data_clean/public_records/`, comme pour la partie Yannis.

In [16]:
for name, df in dfs.items():
    df.to_csv(SORTIE / f"{name}.csv", index=False)
    print(f" Enregistré : {SORTIE}/{name}.csv ({df.shape[0]} lignes, {df.shape[1]} colonnes)")

 Enregistré : data_clean/public_records/institutional_memos.csv (30 lignes, 41 colonnes)
 Enregistré : data_clean/public_records/licensed_facilities.csv (3 lignes, 13 colonnes)
 Enregistré : data_clean/public_records/licensed_professionals.csv (53 lignes, 12 colonnes)
 Enregistré : data_clean/public_records/officer_filings.csv (135 lignes, 19 colonnes)


## Bilan du nettoyage

- Les 9 fichiers sont nettoyés : doublons, espaces, catégories et dates.
- Aucune date n'a été perdue.
- On a ajouté des colonnes utiles pour l'analyse : âges, durées de mandat, mandats en cours, sections des mémos, noms sans titre.
- Les corrections apportées au code d'origine :
  - la variable `re` est renommée `ent` ;
  - l'export de la partie Yannis est activé ;
  - les chemins de la partie Dali sont mis à jour ;
  - la liste des services des établissements reste du texte.

Les 9 fichiers propres sont dans `data_clean/public_records/`.